<a href="https://colab.research.google.com/github/koushik0728/Smart-Traffic-Vehicle-Detection-System/blob/main/Smart_Traffic_Vehicle_Detection_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!pip install --quiet ultralytics opencv-python-headless numpy torch torchvision tqdm matplotlib

In [12]:
import os
import cv2
import numpy as np
import torch
from ultralytics import YOLO
from tqdm import tqdm
import matplotlib.pyplot as plt

from google.colab import files
from IPython.display import HTML, display

In [13]:
# vehicle classes
vehicle_classes = {
    "car": "car",
    "motorcycle": "motorcycle",
    "bus": "bus",
    "truck": "truck",
    "bicycle": "bicycle",
    "auto-rickshaw": "auto-rickshaw",
    "autorickshaw": "auto-rickshaw",
    "rickshaw": "auto-rickshaw",
    "van": "van",
    "tempo": "tempo",
    "tractor": "tractor"
}

# colors for bounding boxes (BGR)
class_colors = {
    "car": (255, 140, 0),
    "motorcycle": (0, 200, 80),
    "bus": (0, 190, 255),
    "truck": (180, 50, 240),
    "bicycle": (220, 220, 0),
    "auto-rickshaw": (30, 100, 255),
    "van": (210, 80, 140),
    "tempo": (140, 180, 50),
    "tractor": (50, 130, 200),
    "default": (200, 200, 200)
}

In [14]:
model_name = "yolov8n.pt"
confidence_threshold = 0.35
iou_threshold = 0.45

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cpu


In [15]:
print(f"Loading YOLO model: {model_name}...")

model = YOLO(model_name)

print("YOLO model loaded successfully.")

target_class_ids = set()
class_id_to_vehicle = {}

for class_id, class_name in model.names.items():
    class_name = str(class_name).strip().lower()

    if class_name in vehicle_classes:
        target_class_ids.add(class_id)
        class_id_to_vehicle[class_id] = vehicle_classes[class_name]

supported_classes = sorted(set(class_id_to_vehicle.values()))

print("Supported vehicle classes:", supported_classes)
print("Vehicle class IDs:", target_class_ids)

Loading YOLO model: yolov8n.pt...
YOLO model loaded successfully.
Supported vehicle classes: ['bicycle', 'bus', 'car', 'motorcycle', 'truck']
Vehicle class IDs: {1, 2, 3, 5, 7}


In [16]:
print("Please upload your traffic video.")

uploaded = files.upload()

if not uploaded:
    raise ValueError("No video was uploaded.")

input_video_path = list(uploaded.keys())[0]

print(f"Video uploaded successfully: {input_video_path}")

Please upload your traffic video.


Saving 14965791_3840_2160_60fps.mp4 to 14965791_3840_2160_60fps.mp4
Video uploaded successfully: 14965791_3840_2160_60fps.mp4


In [17]:
if not os.path.exists(input_video_path):
    raise FileNotFoundError(f"Video not found: {input_video_path}")

cap = cv2.VideoCapture(input_video_path)

if not cap.isOpened():
    raise ValueError(f"Could not open video: {input_video_path}")

video_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
video_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
video_fps = float(cap.get(cv2.CAP_PROP_FPS))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

cap.release()

if video_fps <= 0 or np.isnan(video_fps):
    video_fps = 25.0

video_duration = total_frames / video_fps if total_frames > 0 else 0.0

print("Video information")
print("-" * 40)
print(f"File       : {os.path.basename(input_video_path)}")
print(f"Resolution : {video_width} x {video_height}")
print(f"FPS        : {video_fps:.2f}")
print(f"Frames     : {total_frames}")
print(f"Duration   : {video_duration:.2f} seconds")
print("-" * 40)

Video information
----------------------------------------
File       : 14965791_3840_2160_60fps.mp4
Resolution : 3840 x 2160
FPS        : 60.00
Frames     : 928
Duration   : 15.47 seconds
----------------------------------------


In [18]:
def annotate_frame(frame, detections):
    result = frame.copy()

    for detection in detections:
        x1, y1, x2, y2 = detection["bbox"]
        name = detection["class_name"]
        confidence = detection["confidence"]

        color = class_colors.get(name, class_colors["default"])

        # Draw bounding box
        cv2.rectangle(
            result,
            (x1, y1),
            (x2, y2),
            color,
            2
        )

        # Create label
        confidence_percent = int(confidence * 100)
        label = f"{name} - {confidence_percent}%"

        # Get label size
        (text_width, text_height), _ = cv2.getTextSize(
            label,
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            2
        )

        # Draw label background
        label_y = max(y1 - 10, text_height + 5)

        cv2.rectangle(
            result,
            (x1, label_y - text_height - 5),
            (x1 + text_width + 10, label_y + 5),
            color,
            -1
        )

        # Draw label text
        cv2.putText(
            result,
            label,
            (x1 + 5, label_y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 255, 255),
            2,
            cv2.LINE_AA
        )

    return result

In [19]:
output_video_path = "vehicle_detection_output.mp4"

cap = cv2.VideoCapture(input_video_path)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = float(cap.get(cv2.CAP_PROP_FPS))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

if fps <= 0 or np.isnan(fps):
    fps = 25.0

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(
    output_video_path,
    fourcc,
    fps,
    (width, height)
)

vehicle_counts = {}
frame_count = 0

print(f"Processing video: {input_video_path}")

start_time = time.time()

with tqdm(total=total_frames, desc="Processing video") as progress:

    while cap.isOpened():
        ret, frame = cap.read()

        if not ret:
            break

        frame_count += 1

        results = model.predict(
            frame,
            conf=confidence_threshold,
            iou=iou_threshold,
            device=device,
            verbose=False
        )

        detections = []

        if results and results[0].boxes is not None:
            for box in results[0].boxes:

                class_id = int(box.cls[0].item())

                if class_id not in target_class_ids:
                    continue

                confidence = float(box.conf[0].item())

                x1, y1, x2, y2 = (
                    box.xyxy[0]
                    .cpu()
                    .numpy()
                    .astype(int)
                )

                vehicle_name = class_id_to_vehicle.get(
                    class_id,
                    "vehicle"
                )

                detections.append({
                    "bbox": (x1, y1, x2, y2),
                    "class_name": vehicle_name,
                    "confidence": confidence
                })

                vehicle_counts[vehicle_name] = (
                    vehicle_counts.get(vehicle_name, 0) + 1
                )

        annotated_frame = annotate_frame(frame, detections)

        out.write(annotated_frame)
        progress.update(1)

cap.release()
out.release()

elapsed_time = time.time() - start_time
processing_fps = frame_count / elapsed_time if elapsed_time > 0 else 0

print("\nProcessing complete.")
print(f"Frames processed: {frame_count}")
print(f"Processing time: {elapsed_time:.2f} seconds")
print(f"Average FPS: {processing_fps:.2f}")
print(f"Output video: {output_video_path}")

print("\nVehicle detections:")
for vehicle, count in vehicle_counts.items():
    print(f"{vehicle}: {count}")

Processing video: 14965791_3840_2160_60fps.mp4


Processing video: 100%|██████████| 928/928 [05:11<00:00,  2.98it/s]


Processing complete.
Frames processed: 928
Processing time: 311.17 seconds
Average FPS: 2.98
Output video: vehicle_detection_output.mp4

Vehicle detections:
car: 1836
motorcycle: 1690
truck: 91


In [20]:
print("=" * 60)
print("Vehicle Detection Report")
print("=" * 60)

print(f"Frames processed : {frame_count}")
print(f"Processing time  : {elapsed_time:.2f} seconds")
print(f"Average FPS      : {processing_fps:.2f}")

print("-" * 60)
print("Vehicle detections:")

total_detections = 0

for vehicle, count in sorted(vehicle_counts.items()):
    print(f"{vehicle}: {count}")

    total_detections += count

print("-" * 60)
print(f"Total detections: {total_detections}")
print("=" * 60)

Vehicle Detection Report
Frames processed : 928
Processing time  : 311.17 seconds
Average FPS      : 2.98
------------------------------------------------------------
Vehicle detections:
car: 1836
motorcycle: 1690
truck: 91
------------------------------------------------------------
Total detections: 3617


In [21]:
print("Edge Case 1: Empty Frame")

empty_frame = np.zeros((480, 640, 3), dtype=np.uint8)

detections = []
annotated_frame = annotate_frame(empty_frame, detections)

assert annotated_frame.shape == empty_frame.shape

print("PASS: Empty frame handled correctly.")


print("\nEdge Case 2: Invalid Video")

cap = cv2.VideoCapture("non_existent_file.mp4")

if not cap.isOpened():
    print("PASS: Invalid video was detected correctly.")
else:
    print("WARNING: Invalid video was unexpectedly opened.")

cap.release()

print("\nAll tests completed.")

Edge Case 1: Empty Frame
PASS: Empty frame handled correctly.

Edge Case 2: Invalid Video
PASS: Invalid video was detected correctly.

All tests completed.
